# Phase 4 (comparison): the full grid

Runs the cells of a grid file: six conditions (none, class-weighted, random oversampling, SMOTE, cVAE,
WGAN-GP) x their strengths x the `cnn` and the `rnn` x the grid seeds; 96 runs at 3 seeds. Cells already
done (in git) are skipped, so the grid can be split over several sessions: run until the session
ends, push, and run again.

Before running (details in `kaggle/README.md`):

1. Settings: **Internet on**. Accelerator: **GPU**. For long runs use Save Version > Save & Run All.
2. Add-ons > Secrets: `GITHUB_TOKEN` (Contents: read and write on the repo), ticked for this notebook.
3. Add Input: the team's `mitdb-raw` dataset and the banks dataset; set `BANKS_DATASET_DIR` to its folder.
4. Set the parameters in the next cell (including `GIT_NAME` and `GIT_EMAIL`), then Run All.
5. When it finishes, the ledger and run folders are on GitHub (`git pull`). The checkpoints the final
   evaluation needs are only in the zip in Output: download it and run `python kaggle/ingest_runs.py <zip>`.


In [ ]:
# --- Parameters: edit only this cell ---------------------------------------------------------
VARIANTS = ["full_grid"]              # the grid file(s) in configs/phase4_full_grid/ (full_grid is written by the phase 4a gate)
PREVIEW = False                       # True = run before the gates; runs are marked provisional
REPRESENTATION = ""                   # preview only
BANKS_DATASET_DIR = ""                # the attached Kaggle dataset with the frozen banks (zip of data/synthetic/), e.g. "/kaggle/input/ecgaug-banks"
SMOKE_TEST = False                    # True = quick check on fake data (a minute or two, no results)
GIT_REF = "main"                      # branch name or full commit SHA to run
DEVICE = "auto"                       # "auto" uses the GPU when the session has one
ENV_MODE = "locked"                   # "locked" = exact uv.lock versions; "kaggle" = preinstalled packages (faster, debugging only)
RAW_DATASET_DIR = "/kaggle/input"     # the attached mitdb-raw dataset; "" downloads MIT-BIH from PhysioNet
REPO_URL = "https://github.com/akashsaha477/Gen_Ai_Project.git"
PUSH_RESULTS = True                   # commit the results and push them to GIT_REF (no download or ingest needed)
GIT_NAME = ""                         # your name for the results commit, e.g. "Akash Saha"
GIT_EMAIL = ""                        # your GitHub email for the results commit


In [ ]:
import os, subprocess, sys
from pathlib import Path

WORK = Path("/kaggle/working")
REPO = WORK / "repo"
env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
url = REPO_URL
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    token = None                      # public repository, or the secret is not attached
if token:
    askpass = WORK / ".git-askpass.sh"
    askpass.write_text('#!/bin/sh\necho "$GITHUB_TOKEN"\n')
    askpass.chmod(0o700)
    env.update(GIT_ASKPASS=str(askpass), GITHUB_TOKEN=token)   # the token is never printed or put in a URL
    url = REPO_URL.replace("https://", "https://x-access-token@")

if not REPO.exists():
    subprocess.run(["git", "clone", "--quiet", url, str(REPO)], check=True, env=env)
subprocess.run(["git", "fetch", "--quiet", "origin"], cwd=REPO, check=True, env=env)
ref = GIT_REF if len(GIT_REF) == 40 else f"origin/{GIT_REF}"
subprocess.run(["git", "checkout", "--quiet", "--detach", ref], cwd=REPO, check=True)
sha = subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPO, capture_output=True, text=True).stdout.strip()
print("running commit", sha)

In [ ]:
# Turn the grid names into grid files, e.g. "full_grid" -> configs/phase4_full_grid/full_grid.yaml
configs = []
for letter in VARIANTS:  # grid names
    (path,) = sorted((REPO / "configs/phase4_full_grid").glob(f"{letter}.yaml"))
    configs.append(str(path.relative_to(REPO)))
print("configs:", configs)

extra = ["--device", DEVICE]
if PREVIEW:
    extra.append("--preview")
    if REPRESENTATION:
        extra += ["--representation", REPRESENTATION]
if SMOKE_TEST:
    extra.append("--smoke")
has_raw = RAW_DATASET_DIR and any(Path(RAW_DATASET_DIR).rglob("100.dat"))
cmd = [sys.executable, "kaggle/kaggle_run.py", "--cli", "ecgaug.grid.runner", "--env", ENV_MODE, "--out", str(WORK),
       "--config", *configs]
if has_raw:
    cmd += ["--raw-dir", RAW_DATASET_DIR]
if BANKS_DATASET_DIR:
    cmd += ["--artifacts-dir", BANKS_DATASET_DIR]
if PUSH_RESULTS and not SMOKE_TEST:
    branch = GIT_REF if len(GIT_REF) != 40 else "kaggle-results"
    cmd += ["--push-to", branch, "--git-name", GIT_NAME, "--git-email", GIT_EMAIL]
cmd += ["--", *extra]
code = subprocess.run(cmd, cwd=REPO, env=env).returncode   # env carries the token for the push
print("finished with exit code", code, "(2 = the run was refused, read the message above)")
